# Concurrent calls hold different connections

Two calls must overlap to demonstrate isolation. An event holds each call open until both have borrowed.

Use the repository Python environment and run the cells in order. This notebook has its own setup; no previous notebook is required.

## 1. Import the implementation

The lifecycle code comes from AOA itself. The example supplies only the resource or external connection used to observe it.

In [1]:
import asyncio

from aoa.action_machine.exceptions import HandleError, TransactionLeftOpenError
from aoa.action_machine.resources.connection_scope import connection_scope
from aoa.action_machine.resources.per_call_connection import PerCallConnection
from aoa.action_machine.resources.postgres import PostgresResource

## 2. Define the objects used in this case

The recording connection logs commands and rejects overlapping operations. The pool limits borrowers; neither class parses SQL or stores database rows.

In [2]:
class RecordingConnection:
    """Record commands; do not parse SQL or simulate database storage."""
    def __init__(self, number):
        """Give this connection a visible identity."""
        self.number = number
        self.commands = []
        self.calls = []
        self.busy = False
        self.closed = False
    async def execute(self, query, *params):
        """Reject concurrent use and record one command."""
        if self.busy or self.closed:
            raise RuntimeError("connection is busy or closed")
        self.busy = True
        try:
            await asyncio.sleep(0)
            self.commands.append(query)
            self.calls.append((query, params))
            return "OK"
        finally:
            self.busy = False
    async def close(self):
        """Mark the physical connection closed."""
        self.closed = True

class RecordingPool:
    """Offer a fixed number of observable connections without a server."""
    def __init__(self, size=1):
        """Create the fixed capacity used by this example."""
        self.connections = [RecordingConnection(i + 1) for i in range(size)]
        self.free = asyncio.Queue()
        for connection in self.connections:
            self.free.put_nowait(connection)
        self.held = set()
        self.closed = False
    async def acquire(self, *, timeout=None):
        """Wait for an available connection with the supplied bound."""
        connection = await asyncio.wait_for(self.free.get(), timeout)
        self.held.add(connection)
        return connection
    async def release(self, connection):
        """Return a borrowed connection exactly once."""
        self.held.remove(connection)
        self.free.put_nowait(connection)
    async def close(self):
        """Close the pool only after all borrowers have returned."""
        assert not self.held
        for connection in self.connections:
            await connection.close()
        self.closed = True

class NotesResource(PostgresResource):
    """The notes domain's resource: its own method runs the statement."""

    async def add_note(self, text: str) -> str:
        """Write one note through the driver's statement helper."""
        return await self._execute("INSERT INTO notes(text) VALUES ($1)", (text,))


## 3. Prepare this call

Create the state and declarations used below. Merely declaring a per-call factory does not run it.

In [3]:
pool = RecordingPool(size=2)
ready = asyncio.Event()
arrivals = []
async def work(label):
    """Hold one resource until the other call has also opened."""
    async with connection_scope({"db": PerCallConnection(
        lambda: NotesResource(pool=pool), lambda resource: resource.close(),
    )}) as connections:
        arrivals.append(label)
        if len(arrivals) == 2:
            ready.set()
        await ready.wait()
        db = connections["db"]
        await db.begin()
        await db.add_note(label)
        await db.commit()

## 4. Run and check the observation

The assertions test this case’s stated behavior. Printed output is recorded below; expected failures are caught explicitly.

In [4]:
await asyncio.wait_for(asyncio.gather(work("work A"), work("work B")), 2)
for connection in pool.connections:
    assert connection.commands == ["BEGIN", "INSERT INTO notes(text) VALUES ($1)", "COMMIT"]
    print(connection.number, connection.calls)
assert {connection.calls[1][1] for connection in pool.connections} == {("work A",), ("work B",)}
assert not pool.held
print("borrowed after calls:", len(pool.held))
await pool.close()

1 [('BEGIN', ()), ('INSERT INTO notes(text) VALUES ($1)', ('work A',)), ('COMMIT', ())]
2 [('BEGIN', ()), ('INSERT INTO notes(text) VALUES ($1)', ('work B',)), ('COMMIT', ())]
borrowed after calls: 0


## 5. Explain the result

Each command sequence stayed on one connection. Separate sessions prevent one caller from ending the other caller’s transaction. They do not make shared database rows private or remove the need for database concurrency rules.